# 🧠 The Mind of a Model
# Day 3: Feature Engineering

In [1]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/Aeshwa-Kachhadiya/The-mind-of-a-model/main/dataset/linkedin_post_performance.xlsx'

df = pd.read_excel(url)
print('✅ Dataset loaded!')
print(f'Shape: {df.shape}')
df.head()

✅ Dataset loaded!
Shape: (300, 20)


,word_count,has_image,has_video,has_carousel,has_hashtags,hashtag_count,has_question,has_emoji,topic_category,posting_day,posting_hour,is_weekend,account_followers,account_age_months,posting_frequency,likes_first_hour,comments_first_hour,shares_first_hour,impressions_first_hour,went_viral
0,152.0,Yes,No,No,Yes,6,Yes,Yes,Tech,Wednesday,20,No,5822.0,21,1,185.0,48,21,1889.0,Yes
1,485.0,No,No,No,Yes,8,Yes,No,Motivation,Thursday,10,No,4544.0,59,13,98.0,30,5,3726.0,No
2,398.0,Yes,Yes,No,Yes,7,No,Yes,Tech,Tuesday,12,No,7718.0,56,2,2.0,28,2,116.0,No
3,320.0,Yes,No,No,Yes,9,No,Yes,ML,Saturday,4,Yes,4047.0,22,2,196.0,4,12,4448.0,Yes
4,156.0,No,Yes,Yes,No,0,No,Yes,Personal,Monday,0,No,1776.0,36,3,84.0,24,15,564.0,No


In [2]:
# See what we are working with
print('BEFORE FEATURE ENGINEERING:')
print(df.dtypes)

BEFORE FEATURE ENGINEERING:
word_count                float64
has_image                  object
has_video                  object
has_carousel               object
has_hashtags               object
hashtag_count               int64
has_question               object
has_emoji                  object
topic_category             object
posting_day                object
posting_hour                int64
is_weekend                 object
account_followers         float64
account_age_months          int64
posting_frequency           int64
likes_first_hour          float64
comments_first_hour         int64
shares_first_hour           int64
impressions_first_hour    float64
went_viral                 object
dtype: object


In [3]:
# Encode Yes/No columns to 1/0
# Models only understand numbers
# Yes → 1
# No  → 0

binary_cols = ['has_image', 'has_video', 'has_carousel',
               'has_hashtags', 'has_question', 'has_emoji',
               'is_weekend']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

print('✅ Binary columns encoded!')
print(df[binary_cols].head())

✅ Binary columns encoded!
   has_image  has_video  has_carousel  has_hashtags  has_question  has_emoji  \
0          1          0             0             1             1          1   
1          0          0             0             1             1          0   
2          1          1             0             1             0          1   
3          1          0             0             1             0          1   
4          0          1             1             0             0          1   

   is_weekend  
0           0  
1           0  
2           0  
3           1  
4           0  


In [4]:
# Encode target variable
# Success → 1
# Failed  → 0

df['went_viral'] = df['went_viral'].map({'Yes': 1, 'No': 0})

print('✅ Target variable encoded!')
print(df['went_viral'].value_counts())

✅ Target variable encoded!
went_viral
0    210
1     90
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Encode topic_category
df['topic_category'] = le.fit_transform(df['topic_category'])
print('topic_category encoded:')
print(df['topic_category'].value_counts())

# Encode posting_day
df['posting_day'] = le.fit_transform(df['posting_day'])
print('\nposting_day encoded:')
print(df['posting_day'].value_counts())

topic_category encoded:
topic_category
2    71
4    62
3    62
1    59
0    46
Name: count, dtype: int64

posting_day encoded:
posting_day
3    50
6    45
2    43
1    43
5    41
0    40
4    38
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Scale numerical columns
numerical_cols = ['word_count', 'account_followers',
                  'account_age_months', 'posting_frequency',
                  'likes_first_hour', 'comments_first_hour',
                  'shares_first_hour', 'impressions_first_hour']

# Fill missing values before scaling
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print('✅ Numerical columns scaled!')
print(df[numerical_cols].describe().round(2))

✅ Numerical columns scaled!
       word_count  account_followers  account_age_months  posting_frequency  \
count      300.00             300.00              300.00             300.00   
mean         0.50               0.03                0.53               0.53   
std          0.27               0.07                0.30               0.32   
min          0.00               0.00                0.00               0.00   
25%          0.29               0.01                0.29               0.23   
50%          0.50               0.02                0.55               0.54   
75%          0.73               0.04                0.79               0.85   
max          1.00               1.00                1.00               1.00   

       likes_first_hour  comments_first_hour  shares_first_hour  \
count            300.00               300.00             300.00   
mean               0.02                 0.52               0.50   
std                0.07                 0.30               

/tmp/ipykernel_1741/2248735412.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [7]:
# Create a new feature — engagement_score
# Combining multiple columns into one meaningful feature

df['engagement_score'] = (
    df['likes_first_hour'] +
    df['comments_first_hour'] +
    (df['shares_first_hour'] * 2) +
    (df['impressions_first_hour'] / 100)
)

print('✅ New feature created: engagement_score')
print(df['engagement_score'].describe().round(4))

✅ New feature created: engagement_score
count    300.0000
mean       1.5329
std        0.6300
min        0.0846
25%        1.0663
50%        1.5155
75%        1.9811
max        2.9976
Name: engagement_score, dtype: float64


In [8]:
# Drop columns that add no value to the model

# hashtag_count is already covered by has_hashtags
# posting_hour is already covered by is_weekend and posting_day

cols_to_drop = ['hashtag_count']

df.drop(columns=cols_to_drop, inplace=True)

print('✅ Unnecessary columns dropped!')
print(f'Shape after feature selection: {df.shape}')

✅ Unnecessary columns dropped!
Shape after feature selection: (300, 20)


In [9]:
# Final check
print('AFTER FEATURE ENGINEERING:')
print(f'Shape: {df.shape}')
print()
print('Data types:')
print(df.dtypes)
print()
print('First 5 rows:')
df.head()

AFTER FEATURE ENGINEERING:
Shape: (300, 20)

Data types:
word_count                float64
has_image                   int64
has_video                   int64
has_carousel                int64
has_hashtags                int64
has_question                int64
has_emoji                   int64
topic_category              int64
posting_day                 int64
posting_hour                int64
is_weekend                  int64
account_followers         float64
account_age_months        float64
posting_frequency         float64
likes_first_hour          float64
comments_first_hour       float64
shares_first_hour         float64
impressions_first_hour    float64
went_viral                  int64
engagement_score          float64
dtype: object

First 5 rows:


,word_count,has_image,has_video,has_carousel,has_hashtags,has_question,has_emoji,topic_category,posting_day,posting_hour,is_weekend,account_followers,account_age_months,posting_frequency,likes_first_hour,comments_first_hour,shares_first_hour,impressions_first_hour,went_viral,engagement_score
0,0.225951,1,0,0,1,1,1,4,6,20,0,0.029110,0.344828,0.000000,0.022881,0.979592,0.724138,0.017800,1,2.450926
1,0.970917,0,0,0,1,1,0,2,4,10,0,0.022720,1.000000,0.923077,0.012003,0.612245,0.172414,0.036190,0,0.969437
2,0.776286,1,1,0,1,0,1,4,5,12,0,0.038590,0.948276,0.076923,0.000000,0.571429,0.068966,0.000050,0,0.709360
3,0.601790,1,0,0,1,0,1,1,2,4,1,0.020235,0.362069,0.076923,0.024256,0.081633,0.413793,0.043418,1,0.933909
4,0.234899,0,1,1,0,0,1,3,1,0,0,0.008880,0.603448,0.153846,0.010253,0.489796,0.517241,0.004535,0,1.534577
